In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("Desafio02")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 00:06:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/12 00:06:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df_clientes = spark.read.csv("/workspace/datasets/02_lineage_DAG/clientes.csv", header= True, inferSchema= True)

In [4]:
df_final = (
    df_clientes.filter(
        (F.col("ativo") == True) &
        (F.col("idade") >= 35)
    )
    .select("cliente_id", "nome", "idade", "cidade", "renda_mensal")
    .withColumn("renda_anual", (F.col("renda_mensal")) * 12)
    .withColumn(
        "categoria_renda",
        F.when(F.col("renda_mensal") <= 4000, "Baixa")
        .when((F.col("renda_mensal") > 4000) & (F.col("renda_mensal") <= 8000), "Média")
        .otherwise("Alta")
    )
)


In [5]:
df_final.explain(True)

== Parsed Logical Plan ==
'Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21, renda_anual#35, CASE WHEN ('renda_mensal <= 4000) THEN Baixa WHEN (('renda_mensal > 4000) AND ('renda_mensal <= 8000)) THEN Média ELSE Alta END AS categoria_renda#42]
+- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21, (renda_mensal#21 * cast(12 as double)) AS renda_anual#35]
   +- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21]
      +- Filter ((ativo#22 = true) AND (idade#19 >= 35))
         +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double, renda_anual: double, categoria_renda: string
Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21, renda_anual#35, CASE WHEN (renda_mensal#21 <= cast(4000 as double)) THEN Baixa WHEN ((renda_mensal#21 > cast(4000 as double)) AND (renda_mensal#21 <= 

Pergunta 1

Quais planos aparecem no explain(True)?

Resposta: Planos parcial logico, analtico logico, otimizado logico e fisico

Pergunta 2

Em qual desses planos o Catalyst atua?

Resposta: o Catalyst atua no plano logico otimizando as regras de negocio

Pergunta 3

Qual é a diferença entre:

Parsed Logical Plan
Physical Plan

Resposta: o parsed logical atual apenas traduzindo o codigo e valido a sintaxe basica, o plano fisico define exatamente como o cluster ira executar as operações na pratica

In [6]:
#parte 3
df_final_catalyst = (
    df_clientes
    .withColumn(
            "categoria_renda",
            F.when(F.col("renda_mensal") <= 4000, "Baixa")
            .when((F.col("renda_mensal") > 4000) & (F.col("renda_mensal") <= 8000), "Média")
            .otherwise("Alta")
            )
    .select("cliente_id", "nome", "idade", "cidade", "renda_mensal")
    .withColumn("renda_anual", (F.col("renda_mensal")) * 12)
    .filter(
        (F.col("ativo") == True) &
        (F.col("idade") >= 35)
    )
    
)



In [7]:
df_final_catalyst.explain(True)

== Parsed Logical Plan ==
'Filter (('ativo = true) AND ('idade >= 35))
+- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21, (renda_mensal#21 * cast(12 as double)) AS renda_anual#63]
   +- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21]
      +- Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21, ativo#22, CASE WHEN (renda_mensal#21 <= cast(4000 as double)) THEN Baixa WHEN ((renda_mensal#21 > cast(4000 as double)) AND (renda_mensal#21 <= cast(8000 as double))) THEN Média ELSE Alta END AS categoria_renda#50]
         +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double, renda_anual: double
Project [cliente_id#17, nome#18, idade#19, cidade#20, renda_mensal#21, renda_anual#63]
+- Filter ((ativo#22 = true) AND (idade#19 >= 35))
   +- Project [cliente_id#17, nome#18, idade#19, cidade#20, rend

In [8]:
spark.stop()

Pergunta 4

O Optimized Logical Plan mudou?

Por quê?

Resposta: Não mudou, pois mesmo que eu tenha mudado a ordem do codigo, o spark ainda sim otimiza tudo que foi gerado antes de executar

Pergunta 5

O Physical Plan mudou?

Explique.

Resposta: Permaneceu igual, porque o Catalyst reorganizou o plano lógico de forma que ambos fossem executados da mesma maneira.


Parte 4 — Reflexão (a mais importante)

Sem pesquisar.

Com suas palavras.

Imagine que alguém pergunta:

"Por que o Spark não executa cada transformação imediatamente?"

Responda como se estivesse explicando para um colega que nunca utilizou Spark.

Resposta: Eu faria a analagia de um supermercado, assim que entramos no supermecardo, fazemos um planejamento em quais corredores vamos passar e pegar tudo de necessaria em cada corredor, esse planejamento seria o catalyst, e realmente quando estivermos nos corredores pegando os itens, seria uma action
